# Analiza amplitude kalcijumskog odgovora ćelija na tretman kalijumom

## Uvodne napomene

### Cilj

U ovom notebook-u, biće izvršena analiza amplitude odgovora ćelija na tretman kalijumom u vidu porasta unutarćelijske koncentracije kalcijuma, u svrhu karakterisanja raspodele intenziteta ovog odgovora i potencijalnog definisanja njegovog minimalnog intenziteta za posmatranje ćelija kao fiziološki adekvatnih i upotrebljivih u analizi.

Za analizu će se iskoristiti rezultati isključivo onih eksperimenata u kojima su ćelije tretirane prvo rastvorom bogatim kalijumom, bez prethodnog tretmana CSF-om ili blokatorima kanala.

Glavna početna pretpostavka je da se u eksperimentima mogu identifikovati dve subpopulacije ćelija - one koje pokazuju zadovoljavajuć odgovor na K+ i koje se stoga mogu koristiti za pouzdanu analizu njihovog odgovora na CSF, kao i one koje imaju nekarakteristično slab odgovor na K+, što sugeriše da su oštećene ili im je fiziologija na drugi način poremećena, i koje zato treba izbaciti iz analiza.

## Učitavanje podataka

In [1]:
from pathlib import Path, PurePosixPath
import re
import shutil
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Definisanje strukture projekta
RANDOM_STATE = 42

def find_project_root(
    start: Path | None = None,
) -> Path:
    """
    Locira osnovni folder projekta pretragom trenutne direktorije
    i foldera višeg reda.
    """
    start = (
        Path.cwd()
        if start is None
        else Path(start)
    ).resolve()

    for candidate in [
        start,
        *start.parents,
    ]:
        if (
            (candidate / "environment.yml").is_file()
            and (candidate / "notebooks").is_dir()
            and (candidate / "testing").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Osnovni folder nije pronađen. Uverite se da "
        "environment.yml, notebooks/ i testing/ postoje "
        "u glavnom folderu projekta."
    )


PROJECT_ROOT = find_project_root()

K_ANALYSIS_ROOT = (
    PROJECT_ROOT
    / "testing"
    / "k_amplitude_testing"
)

K_DATA_DIR = (
    K_ANALYSIS_ROOT
    / "data"
)

K_RAW_DIR = (
    K_DATA_DIR
    / "raw"
)

K_UNPACKED_DIR = (
    K_DATA_DIR
    / "unpacked"
)

K_EXTRACTED_DIR = (
    K_DATA_DIR
    / "extracted"
)

K_RESULTS_DIR = (
    K_ANALYSIS_ROOT
    / "results"
)

K_FIGURES_DIR = (
    K_RESULTS_DIR
    / "figures"
)

K_TABLES_DIR = (
    K_RESULTS_DIR
    / "tables"
)

K_ZIP_PATH = (
    K_RAW_DIR
    / "all_dilutions.zip"
)

for directory in [
    K_RAW_DIR,
    K_UNPACKED_DIR,
    K_EXTRACTED_DIR,
    K_FIGURES_DIR,
    K_TABLES_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

if not K_ZIP_PATH.is_file():
    raise FileNotFoundError(
        "Arhiva sa Excel fajlovima eksperimenata nije pronađena:\n"
        f"{K_ZIP_PATH}"
    )

In [3]:
# Obrazac imena foldera
K_SOURCE_FOLDER_PATTERN = re.compile(
    r"^(?P<patient>[A-Za-z0-9]+)_1\.\.(?P<denominator>\d+)$"
)

# Klase bolesti po pacijentu
K_CLASS_MAP = {
    "3": "ALS",
    "4": "ALS",
    "7": "Control",
    "10a": "SMA",
    "10b": "SMA",
}

def unpack_k_amplitude_archive(
    zip_path: Path,
    unpacked_directory: Path,
    clear_existing: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Ekstrahuje Excel fajlove iz arhive i organizuje ih prema razblaženju.

    Primer izvornog foldera:
        10a_1..25

    Izlaz:
        data/unpacked/1_25/10a_1..25/file.xlsx

    Vraća:
        manifest:
            Jedan red po ekstrahovanom fajlu.

        skipped:
            Fajlovi iz arhive koji su ignorisani ili se nisu mogli
            dodeliti odgovarajućem folderu.
    """
    if not zip_path.is_file():
        raise FileNotFoundError(
            f"Arhiva nije pronađena: {zip_path}"
        )

    if clear_existing and unpacked_directory.exists():
        shutil.rmtree(
            unpacked_directory
        )

    unpacked_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    manifest_rows = []
    skipped_rows = []

    with zipfile.ZipFile(
        zip_path,
        mode="r",
    ) as archive:

        for member in archive.infolist():
            if member.is_dir():
                continue

            member_path = PurePosixPath(
                member.filename.replace(
                    "\\",
                    "/",
                )
            )

            if (
                "__MACOSX" in member_path.parts
                or any(
                    part.startswith(".")
                    for part in member_path.parts
                )
            ):
                skipped_rows.append(
                    {
                        "zip_member": member.filename,
                        "reason": "hidden_or_system_file",
                    }
                )
                continue

            if member_path.name.startswith("~$"):
                skipped_rows.append(
                    {
                        "zip_member": member.filename,
                        "reason": "temporary_excel_file",
                    }
                )
                continue

            if member_path.suffix.lower() != ".xlsx":
                skipped_rows.append(
                    {
                        "zip_member": member.filename,
                        "reason": "not_xlsx",
                    }
                )
                continue

            folder_matches = []

            for part in member_path.parts[:-1]:
                match = (
                    K_SOURCE_FOLDER_PATTERN
                    .fullmatch(part)
                )

                if match is not None:
                    folder_matches.append(
                        (
                            part,
                            match,
                        )
                    )

            if len(folder_matches) != 1:
                skipped_rows.append(
                    {
                        "zip_member": member.filename,
                        "reason": (
                            "experiment_folder_not_identified"
                        ),
                    }
                )
                continue

            source_folder, folder_match = (
                folder_matches[0]
            )

            patient_id = (
                folder_match.group(
                    "patient"
                )
            )

            denominator = int(
                folder_match.group(
                    "denominator"
                )
            )

            dilution = f"1/{denominator}"
            dilution_folder = (
                f"1_{denominator}"
            )

            destination_path = (
                unpacked_directory
                / dilution_folder
                / source_folder
                / member_path.name
            )

            destination_path.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            with archive.open(
                member,
                mode="r",
            ) as source_file:
                with destination_path.open(
                    mode="wb",
                ) as destination_file:
                    shutil.copyfileobj(
                        source_file,
                        destination_file,
                    )

            manifest_rows.append(
                {
                    "zip_member": member.filename,
                    "source_folder": source_folder,
                    "patient_id": patient_id,
                    "class_label": K_CLASS_MAP.get(
                        patient_id,
                        "Unknown",
                    ),
                    "dilution": dilution,
                    "dilution_folder": (
                        dilution_folder
                    ),
                    "file_name": (
                        member_path.name
                    ),
                    "unpacked_relative_path": str(
                        destination_path.relative_to(
                            unpacked_directory
                        )
                    ),
                }
            )

    manifest = pd.DataFrame(
        manifest_rows
    )

    skipped = pd.DataFrame(
        skipped_rows,
        columns=[
            "zip_member",
            "reason",
        ],
    )

    if manifest.empty:
        raise RuntimeError(
            "Excel fajlovi nisu ekstrahovani. "
            "Proverite imena foldera u ZIP fajlu."
        )

    return manifest, skipped

In [4]:
def save_figure(
    fig,
    file_stem: str,
    dpi: int = 300,
) -> None:
    """
    Čuva grafik K+ analize kao PNG i PDF.
    """
    png_path = (
        K_FIGURES_DIR
        / f"{file_stem}.png"
    )

    pdf_path = (
        K_FIGURES_DIR
        / f"{file_stem}.pdf"
    )

    fig.savefig(
        png_path,
        dpi=dpi,
        bbox_inches="tight",
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
    )

In [5]:
# Ekstrakcija fajlova i kreiranje manifesta eksperimenata
workbook_manifest, skipped_zip_members = (
    unpack_k_amplitude_archive(
        zip_path=K_ZIP_PATH,
        unpacked_directory=(
            K_UNPACKED_DIR
        ),
        clear_existing=True,
    )
)

workbook_manifest.to_csv(
    K_EXTRACTED_DIR
    / "k_workbook_manifest.csv",
    index=False,
)

skipped_zip_members.to_csv(
    K_EXTRACTED_DIR
    / "k_skipped_zip_members.csv",
    index=False,
)

print("Ekstrahovani fajlovi:", len(workbook_manifest))

display(
    workbook_manifest
    .groupby(
        [
            "dilution",
            "class_label",
        ],
        dropna=False,
    )
    .size()
    .rename("n_files")
    .reset_index()
)

if not skipped_zip_members.empty:
    print(
        "\nPreskočeni fajlovi iz arhive:",
        len(skipped_zip_members),
    )

    display(
        skipped_zip_members
    )

Ekstrahovani fajlovi: 29


,dilution,class_label,n_files
0,1/10,ALS,5
1,1/10,Control,4
2,1/10,SMA,2
3,1/100,SMA,4
4,1/25,ALS,4
5,1/25,Control,2
6,1/25,SMA,3
7,1/50,ALS,1
8,1/50,SMA,4


In [6]:
# Pregled otpakovanih fajlova
xlsx_files = sorted(
    path
    for path in K_UNPACKED_DIR.rglob(
        "*.xlsx"
    )
    if not path.name.startswith("~$")
)

if not xlsx_files:
    raise FileNotFoundError(
        "Nisu pronađeni Excel fajlovi u:\n"
        f"{K_UNPACKED_DIR}"
    )

print(f"Identifikovano {len(xlsx_files)} Excel fajlova:")

for path in xlsx_files:
    print(
        path.relative_to(
            K_UNPACKED_DIR
        )
    )

Identifikovano 29 Excel fajlova:
1_10/10a_1..10/2023_11_01_CGNs_isolated3010_CSF_1.xlsx
1_10/10a_1..10/2023_11_01_CGNs_isolated3010_CSF_2.xlsx
1_10/3_1..10/2024_02_16_CGNs_iso1302_1_min0.1.xlsx
1_10/4_1..10/2023_11_11_CGNs_isolated0911_CSF_2_min0.1.xlsx
1_10/4_1..10/2023_11_11_CGNs_isolated0911_CSF_3_min0.1.xlsx
1_10/4_1..10/2023_11_11_CGNs_isolated0911_CSF_4_min0.1.xlsx
1_10/4_1..10/2024_02_04_CGNs_isolated0202_CSF_1.xlsx
1_10/7_1..10/2023_11_17_CGNs_isolated1511_CSF_3_min0.1.xlsx
1_10/7_1..10/2024_03_15_CGNs_isolated1203_CSF_1_min0.1.xlsx
1_10/7_1..10/2024_03_15_CGNs_isolated1203_CSF_4_min0.1.xlsx
1_10/7_1..10/2024_03_15_CGNs_isolated1203_CSF_6_min0.1.xlsx
1_100/10a_1..100/2023_11_02_CGNs_isolated3010_CSF_4.xlsx
1_100/10a_1..100/2023_11_02_CGNs_isolated3010_CSF_5.xlsx
1_100/10a_1..100/2023_11_02_CGNs_isolated3010_CSF_6_min0.1.xlsx
1_100/10a_1..100/2023_11_02_CGNs_isolated3010_CSF_7.xlsx
1_25/10a_1..25/2023_11_01_CGNs_isolated3010_CSF_3.xlsx
1_25/10a_1..25/2023_11_01_CGNs_isolated3010

## Ekstrakcija vrednosti

In [7]:
def excel_column_letter(column_number: int) -> str:
    """
    Prevodi kolonu Excel fajla u slovni oblik.

    Primer:
        1  -> A
        2  -> B
        26 -> Z
        27 -> AA
    """
    letters = ""

    while column_number:
        column_number, remainder = divmod(column_number - 1, 26)
        letters = chr(65 + remainder) + letters

    return letters

In [8]:
def normalize_label(value) -> str:
    """
    Konvertuje oznaku lista u standardizovan niz sa malim slovima.

    Ovako se izbegavaju problemi u poređenju oznaka poput
    'prvi kalijum', 'Prvi kalijum ', itd.
    """
    if pd.isna(value):
        return ""

    return str(value).strip().lower()

In [9]:
def parse_folder_metadata(
    path: Path,
) -> dict:
    """
    Određuje ID pacijenta, razblaženje i klasu bolesti za dati fajl
    na osnovu foldera u kom se nalazi.
    
    Očekivana struktura direktorije:
        data/unpacked/
            1_25/
                10a_1..25/
                    file.xlsx
    """
    relative_path = path.relative_to(
        K_UNPACKED_DIR
    )

    if len(relative_path.parts) < 3:
        raise ValueError(
            "Neočekivana struktura direktorije:\n"
            f"{relative_path}"
        )

    dilution_folder = (
        relative_path.parts[0]
    )

    folder_name = (
        relative_path.parts[-2]
    )

    folder_match = (
        K_SOURCE_FOLDER_PATTERN
        .fullmatch(folder_name)
    )

    if folder_match is None:
        raise ValueError(
            "Naziv foldera se ne poklapa sa očekivanim obrascem: "
            f"{folder_name}"
        )

    patient_id = (
        folder_match.group(
            "patient"
        )
    )

    denominator = int(
        folder_match.group(
            "denominator"
        )
    )

    dilution = f"1/{denominator}"

    return {
        "folder_name": folder_name,
        "patient_id": patient_id,
        "dilution": dilution,
        "dilution_folder": (
            dilution_folder
        ),
        "class_label": K_CLASS_MAP.get(
            patient_id,
            "Unknown",
        ),
    }

In [10]:
def get_sheet_first_cells(path: Path) -> pd.DataFrame:
    """
    Prolazi kroz sve listove u fajlu i vraća oznake njihovih A1 ćelija.

    Radi i sa praznim listovima ili onim gde je nemoguće pročitati oznaku.
    """
    excel_file = pd.ExcelFile(path)

    rows = []

    for sheet_index, sheet_name in enumerate(
        excel_file.sheet_names
    ):
        try:
            first_cell_table = pd.read_excel(
                path,
                sheet_name=sheet_name,
                header=None,
                nrows=1,
                usecols=[0],
            )

            if first_cell_table.empty:
                first_cell = np.nan
            else:
                first_cell = first_cell_table.iloc[0, 0]

        except Exception as error:
            first_cell = np.nan

        rows.append(
            {
                "sheet_index": sheet_index,
                "sheet_name": sheet_name,
                "first_cell": first_cell,
                "label_normalized": normalize_label(first_cell),
            }
        )

    return pd.DataFrame(rows)

In [11]:
def detect_first_potassium_sheet(path: Path) -> dict:
    """
    Pronalazi list u kom se nalaze parametri prvog K+ odgovora.

    List mora sadržati 'prvi' i 'kalijum'.
    """
    sheet_labels = get_sheet_first_cells(path)

    labels = sheet_labels["label_normalized"]

    potassium_mask = (
        labels.str.contains("kalijum", regex=False)
    )

    potassium_sheets = sheet_labels.loc[
        potassium_mask
    ].copy()

    if potassium_sheets.empty:
        raise ValueError(
            f"Nema lista s prvim K+ odgovorom u {path.name}"
        )

    first_k_mask = (
        potassium_sheets["label_normalized"]
        .str.contains("prvi", regex=False)
    )

    first_k_sheets = potassium_sheets.loc[
        first_k_mask
    ].copy()

    if not first_k_sheets.empty:
        selected = first_k_sheets.sort_values(
            "sheet_index"
        ).iloc[0]

    return {
        "k_sheet_name": selected["sheet_name"],
        "k_sheet_index": selected["sheet_index"],
        "k_sheet_label": selected["first_cell"],
    }

In [12]:
def safe_summary_row(
    table: pd.DataFrame,
    row_index: int,
    roi_columns,
) -> pd.Series:
    """
    Izvlači vrednosti jednog reda unutar lista s parametrima K+ odgovora
    i konvertuje ih u numeričke.
    """
    if row_index >= len(table):
        return pd.Series(
            np.nan,
            index=roi_columns,
        )

    return pd.to_numeric(
        table.iloc[row_index, 1:],
        errors="coerce",
    )

In [13]:
def extract_first_k_summary(
    path: Path,
) -> pd.DataFrame:
    """
    Izvlači parametre prvog odgovora na K+ iz jednog Excel fajla.
    """
    metadata = parse_folder_metadata(
        path
    )

    k_sheet_info = (
        detect_first_potassium_sheet(
            path
        )
    )

    table = pd.read_excel(
        path,
        sheet_name=(
            k_sheet_info[
                "k_sheet_name"
            ]
        ),
        header=None,
    )

    roi_columns = table.columns[1:]
    n_rois = len(roi_columns)

    result = pd.DataFrame(
        {
            "roi_position": np.arange(
                1,
                n_rois + 1,
            ),
            "roi_id": [
                f"ROI_{index:03d}"
                for index in range(
                    1,
                    n_rois + 1,
                )
            ],
            "roi_source_excel_column": [
                excel_column_letter(
                    index + 1
                )
                for index in range(
                    1,
                    n_rois + 1,
                )
            ],
        }
    )

    result["k_ref_amplitude"] = (
        safe_summary_row(
            table,
            13,
            roi_columns,
        ).to_numpy()
    )

    result["k_half_width"] = (
        safe_summary_row(
            table,
            24,
            roi_columns,
        ).to_numpy()
    )

    result["k_peak_location"] = (
        safe_summary_row(
            table,
            35,
            roi_columns,
        ).to_numpy()
    )

    result["k_integral"] = (
        safe_summary_row(
            table,
            46,
            roi_columns,
        ).to_numpy()
    )

    result["k_response_detected"] = (
        result[
            "k_ref_amplitude"
        ].notna()
    )

    for key, value in metadata.items():
        result[key] = value

    result["file_name"] = path.name

    result["relative_path"] = str(
        path.relative_to(
            K_UNPACKED_DIR
        )
    )

    for key, value in (
        k_sheet_info.items()
    ):
        result[key] = value

    return result

In [14]:
def extract_trace_qc(path: Path) -> pd.DataFrame:
    """
    Računa udeo negativnih vrednosti u kolonama lista sa normalizovanim
    podacima o intenzitetu fluorescencije u ćelijama.
    """
    table = pd.read_excel(
        path,
        sheet_name="Sheet2",
        header=None,
    )

    roi_columns = table.columns[1:]
    n_rois = len(roi_columns)

    traces = table.iloc[:, 1:].apply(
        pd.to_numeric,
        errors="coerce",
    )

    n_valid_points = traces.notna().sum(axis=0)

    negative_fraction = (
        (traces < 0).sum(axis=0)
        / n_valid_points.replace(0, np.nan)
    )

    qc = pd.DataFrame(
        {
            "roi_position": np.arange(1, n_rois + 1),
            "roi_id": [
                f"ROI_{i:03d}"
                for i in range(1, n_rois + 1)
            ],
            "roi_source_excel_column": [
                excel_column_letter(i + 1)
                for i in range(1, n_rois + 1)
            ],
            "normalized_negative_fraction": (
                negative_fraction.to_numpy()
            ),
        }
    )

    qc["majority_negative_trace"] = (
        qc["normalized_negative_fraction"] > 0.5
    )

    return qc

In [ ]:
# Ekstrakcija iz svih fajlova
all_k_tables = []
failed_file_rows = []

for path in xlsx_files:
    try:
        k_table = (
            extract_first_k_summary(
                path
            )
        )

        try:
            trace_qc = extract_trace_qc(
                path
            )

            k_table = k_table.merge(
                trace_qc,
                on=[
                    "roi_position",
                    "roi_id",
                    (
                        "roi_source_"
                        "excel_column"
                    ),
                ],
                how="left",
                validate="one_to_one",
            )

        except Exception as qc_error:
            print(
                "trace_qc greška:\n"
                f"{path.name}: {qc_error}"
            )

            k_table[
                "normalized_negative_fraction"
            ] = np.nan

            k_table[
                "majority_negative_trace"
            ] = np.nan

        all_k_tables.append(
            k_table
        )

    except Exception as extraction_error:
        failed_file_rows.append(
            {
                "relative_path": str(
                    path.relative_to(
                        K_UNPACKED_DIR
                    )
                ),
                "error_type": type(
                    extraction_error
                ).__name__,
                "error": str(
                    extraction_error
                ),
            }
        )

k_all = pd.concat(
    all_k_tables,
    ignore_index=True,
)

failed_files = pd.DataFrame(
    failed_file_rows,
    columns=[
        "relative_path",
        "error_type",
        "error",
    ],
)

print("Dimenzije ekstrahovane K+ tabele:", k_all.shape)

print("Neuspelo fajlova:", len(failed_files))

In [ ]:
# Kreiranje jedinstvenih oznaka
k_all["experiment_source_id"] = (
    k_all["folder_name"].astype(str)
    + "__"
    + k_all["file_name"].astype(str)
)

k_all["record_id"] = (
    k_all["experiment_source_id"]
    + "__"
    + k_all["roi_id"]
)

# Obeležavanje ćelija s validnim signalom
k_all["majority_negative_trace"] = (
    k_all["majority_negative_trace"]
    .fillna(False)
    .astype(bool)
)

In [ ]:
# Provera ekstrahovanih listova
selected_k_sheets = (
    k_all[
        [
            "experiment_source_id",
            "dilution_folder",
            "folder_name",
            "patient_id",
            "dilution",
            "class_label",
            "file_name",
            "relative_path",
            "k_sheet_name",
            "k_sheet_label",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "dilution",
            "folder_name",
            "file_name",
        ]
    )
    .reset_index(drop=True)
)

In [ ]:
# Čuvanje tabele sa svim ROI
K_ALL_ROIS_PATH = (
    K_EXTRACTED_DIR
    / "k_first_response_all_rois.csv"
)

k_all.to_csv(
    K_ALL_ROIS_PATH,
    index=False,
)

print("Saved:", K_ALL_ROIS_PATH)

In [ ]:
# Filtriranje ROI sa K+ odgovorom
k_amp_df = k_all.loc[
    k_all["k_response_detected"]
    & k_all["k_ref_amplitude"].notna()
].copy()

# Čuvanje početne tabele za analizu
K_ANALYSIS_INPUT_PATH = (
    K_EXTRACTED_DIR
    / "k_first_amplitude_analysis_input.csv"
)

k_amp_df.to_csv(
    K_ANALYSIS_INPUT_PATH,
    index=False,
)

print("Ukupno ROI:", len(k_all))
print("ROI sa K+ amplitudom:", len(k_amp_df))

## Analiza raspodele amplitude

In [ ]:
# Sažetak po fajlu i folderu
analysis_df = pd.read_csv(
    K_ANALYSIS_INPUT_PATH
)

k_file_summary = (
    analysis_df
    .groupby(
        [
            "folder_name",
            "patient_id",
            "dilution",
            "class_label",
            "file_name",
        ],
        dropna=False,
    )
    .agg(
        n_rois=("record_id", "count"),
        median_k_ref_amplitude=("k_ref_amplitude", "median"),
        mean_k_ref_amplitude=("k_ref_amplitude", "mean"),
        min_k_ref_amplitude=("k_ref_amplitude", "min"),
        max_k_ref_amplitude=("k_ref_amplitude", "max"),
    )
    .reset_index()
)

k_file_summary.sort_values("median_k_ref_amplitude")

In [ ]:
# Sažetak po razblaženju i klasi
k_dilution_summary = (
    analysis_df
    .groupby(
        [
            "dilution",
            "class_label",
        ],
        dropna=False,
    )
    .agg(
        n_rois=("record_id", "count"),
        median_k_ref_amplitude=("k_ref_amplitude", "median"),
        mean_k_ref_amplitude=("k_ref_amplitude", "mean"),
        min_k_ref_amplitude=("k_ref_amplitude", "min"),
        max_k_ref_amplitude=("k_ref_amplitude", "max"),
    )
    .reset_index()
)

k_dilution_summary

In [ ]:
# Prikaz amplitude po postotcima
k_ref_amplitude_summary = analysis_df["k_ref_amplitude"].describe(
    percentiles=[
        0.01,
        0.025,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.975,
        0.99,
    ]
)

k_ref_amplitude_summary

In [ ]:
# Vizuelizacija raspodele amplitude
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(
    analysis_df["k_ref_amplitude"],
    bins=40,
    edgecolor="black",
)

ax.set_xlabel("K+ amplituda")
ax.set_ylabel("Broj ROI")
ax.set_title("Raspodela K+ amplitude (svi eksperimenti)")

fig.tight_layout()

save_figure(
    fig,
    "k_amplitude_distribution",
)

plt.show()

In [ ]:
# Log-transformisana raspodela
analysis_df["log_k_ref_amplitude"] = np.log(
    analysis_df["k_ref_amplitude"]
)

plt.figure(figsize=(7, 4))

plt.hist(
    analysis_df["log_k_ref_amplitude"],
    bins=40,
    edgecolor="black",
)

plt.xlabel("log(K+ amplituda)")
plt.ylabel("Broj ROI")
plt.title("Log-transformisana raspodela K+ amplitude")
plt.tight_layout()
plt.show()

In [ ]:
# Boksplot grafik po eksperimentu

# Kraće oznake eksperimenata
analysis_df["experiment_label"] = (
    analysis_df["folder_name"].astype(str)
    + " | "
    + analysis_df["file_name"]
    .str.replace(".xlsx", "", regex=False)
)

# Definisanje redosleda
group_column = "experiment_label"

experiment_order = (
    analysis_df
    .groupby("experiment_label")["k_ref_amplitude"]
    .median()
    .sort_values()
    .index
    .tolist()
)

# Vizuelizacija
data_to_plot = [
    analysis_df.loc[
        analysis_df["experiment_label"] == experiment,
        "k_ref_amplitude",
    ].dropna()
    for experiment in experiment_order
]

fig, ax = plt.subplots(figsize=(12, 10))

ax.boxplot(
    data_to_plot,
    tick_labels=experiment_order,
    orientation='horizontal',
)

ax.set_xlabel("K+ amplituda")
ax.set_ylabel("Pojedinačni eksperiment")
ax.set_title("K+ amplituda po eksperimentu")

fig.tight_layout()

save_figure(
    fig,
    "k_amplitude_by_experiment",
)

plt.show()

In [ ]:
# Kumulativna raspodela
values = np.sort(
    analysis_df["k_ref_amplitude"].dropna().to_numpy()
)

cdf = np.arange(1, len(values) + 1) / len(values)

plt.figure(figsize=(7, 4))

plt.plot(values, cdf)

plt.xlabel("K+ amplituda")
plt.ylabel("Udeo ROI")
plt.title("CDF K+ amplitude")
plt.tight_layout()
plt.show()

Ono što zasad vidimo jeste da je opseg vrednosti K+ amplitude veoma veliki kada su u pitanju pojedinačne ćelije, ali i kada su u pitanju odvojeni eksperimenti, sa medijalnim amplitudama po eksperimentu u opsegu od `1.1` do `5.7`, pri čemu je ukupna medijalna vrednost `3.5`. Na osnovu histograma ne primećujemo jasno definisane subpopulacije ćelija, ali ovo možemo dalje ispitati; na primer, pomoću *Gaussian mixture* modela, koji može ukazati na postojanje više preklapajućih populacija ćelija sa normalnom raspodelom amplitude. 

Pre toga ćemo brzo ispitati nekoliko potencijalnih graničnih vrednosti za K+ amplitudu i proveriti šta bi one značile za celokupan set podataka. 

In [ ]:
# Potencijalne granice odgovora
q01 = analysis_df["k_ref_amplitude"].quantile(0.01)
q025 = analysis_df["k_ref_amplitude"].quantile(0.025)
q05 = analysis_df["k_ref_amplitude"].quantile(0.05)
q10 = analysis_df["k_ref_amplitude"].quantile(0.10)
q15 = analysis_df["k_ref_amplitude"].quantile(0.15)

candidate_thresholds = pd.DataFrame(
    {
        "threshold_name": [
            "1. postotak",
            "2.5. postotak",
            "5. postotak",
            "10. postotak",
            "15. postotak",
        ],
        "k_ref_amplitude_threshold": [
            q01,
            q025,
            q05,
            q10,
            q15,
        ],
        "n_flagged": [
            (analysis_df["k_ref_amplitude"] < q01).sum(),
            (analysis_df["k_ref_amplitude"] < q025).sum(),
            (analysis_df["k_ref_amplitude"] < q05).sum(),
            (analysis_df["k_ref_amplitude"] < q10).sum(),
            (analysis_df["k_ref_amplitude"] < q15).sum(),
        ],
    }
)

candidate_thresholds

Vidimo kolika bi bila granična vrednost amplitude za različite postotke ukupnog seta podataka. Međutim, ovo suštinski nije od značaja, jer su ove potencijalne granice sasvim arbitrarne i (barem naizgled) ne odgovaraju nikakvim biološki značajnim razlikama.

### Modeliranje multimodalnosti

Potencijalnu multimodalnost isprobaćemo preko modela Gausove mešavine, koji se koristi za ispitivanje postojanja više preklapajućih normalnih raspodela u okviru veće raspodele. Performanse ovakvog modela procenjuju se preko **BIC** i **AIC** vrednosti, koje su niže ukoliko se model sa zadatim brojem komponenti bolje poklapaju sa podacima, čak i kada se model penalizuje usled povećane složenosti.

In [ ]:
from sklearn.mixture import GaussianMixture

In [ ]:
# Kreiranje Gaussian Mixture modela
X_log_k = (
    analysis_df["log_k_ref_amplitude"]
    .dropna()
    .to_numpy()
    .reshape(-1, 1)
)

gmm_rows = []

for n_components in [1, 2, 3]:
    gmm = GaussianMixture(
        n_components=n_components,
        random_state=RANDOM_STATE,
    )

    gmm.fit(X_log_k)

    gmm_rows.append(
        {
            "n_components": n_components,
            "bic": gmm.bic(X_log_k),
            "aic": gmm.aic(X_log_k),
        }
    )

gmm_comparison = pd.DataFrame(gmm_rows)

gmm_comparison

Vidimo da model sa dve komponente ima znatno bolje performanse od modela sa samo jednom komponentom, dok je model sa tri komponente nešto bolji od onog sa dve; ipak, za potrebe naše analize, modeliranje tri ili više potencijalnih populacija ćelija nije od preterane koristi. Istovremeno, ono što je veoma značajno jeste činjenica da posmatramo sve ćelije odjednom, a već smo videli znatne razlike u raspodelama vrednosti K+ amplitude među eksperimentima; stoga, bilo kakve identifikovane subpopulacije ćelija verovatno ne odgovaraju grupama sa biološki relevantnim razlikama, već najpre potiču od razlika specifičnih za eksperiment.

Za sada ćemo fitovati dvokomponentni model na naše podatke u svrhu ispitivanja postojanja dve manje ili više dobro definisane subpopulacije ćelija.

In [ ]:
# Fitovanje dvokomponentnog modela
gmm_2 = GaussianMixture(
    n_components=2,
    random_state=RANDOM_STATE,
)

gmm_2.fit(X_log_k)

In [ ]:
# Kriva gustine
x_grid = np.linspace(
    X_log_k.min(),
    X_log_k.max(),
    1000,
).reshape(-1, 1)

component_means = gmm_2.means_.ravel()
component_vars = gmm_2.covariances_.ravel()
component_sds = np.sqrt(component_vars)
component_weights = gmm_2.weights_.ravel()

component_densities = []

for mean, sd, weight in zip(
    component_means,
    component_sds,
    component_weights,
):
    density = (
        weight
        * (1 / (sd * np.sqrt(2 * np.pi)))
        * np.exp(
            -0.5
            * ((x_grid.ravel() - mean) / sd) ** 2
        )
    )

    component_densities.append(density)

total_density = np.sum(
    component_densities,
    axis=0,
)

In [ ]:
# Vizuelizacija
fig, ax = plt.subplots(figsize=(7,4))

ax.hist(
    X_log_k.ravel(),
    bins=40,
    density=True,
    alpha=0.5,
    edgecolor="black",
    label="Logaritam zabeleženih amplituda",
)

ax.plot(
    x_grid.ravel(),
    total_density,
    linewidth=2,
    label="Dvokomponentni GMM",
)

for i, density in enumerate(component_densities):
    ax.plot(
        x_grid.ravel(),
        density,
        linestyle="--",
        label=f"Komponenta {i + 1}",
    )

ax.set_xlabel("log(K+ amplituda)")
ax.set_ylabel("Gustina")
ax.set_title("Model raspodele K+ amplitude kao Gausove mešavine")
ax.legend()
fig.tight_layout()

save_figure(
    fig,
    "gmm_absolute_k_amplitude_distribution",
)

plt.show()

In [ ]:
# Identifikacija komponente sa slabim odgovorom
low_component = np.argmin(
    gmm_2.means_.ravel()
)

component_probabilities = gmm_2.predict_proba(X_log_k)

low_component_probability = (
    component_probabilities[:, low_component]
)

# Dodavanje vrednosti u tabelu
analysis_df = analysis_df.loc[
    analysis_df["log_k_ref_amplitude"].notna()
].copy()

analysis_df["weak_k_component_probability"] = (
    low_component_probability
)

analysis_df["weak_k_component_flag_90"] = (
    analysis_df["weak_k_component_probability"] >= 0.90
)

analysis_df["weak_k_component_flag_75"] = (
    analysis_df["weak_k_component_probability"] >= 0.75
)

In [ ]:
# Provera raspodele verovatnoće pripadanja slaboj komponenti
analysis_df[
    [
        "weak_k_component_probability",
    ]
].describe()

In [ ]:
# Broj ROI s datom verovatnoćom pripadanja slaboj komponenti
analysis_df[
    [
        "weak_k_component_flag_90",
        "weak_k_component_flag_75",
    ]
].sum()

In [ ]:
# Procena granične vrednosti
grid_probabilities = gmm_2.predict_proba(x_grid)

low_component_grid_probability = (
    grid_probabilities[:, low_component]
)

# Tačka sa 50% verovatnoćom pripadanja obema komponentama
threshold_index = np.argmin(
    np.abs(low_component_grid_probability - 0.5)
)

log_threshold = x_grid.ravel()[threshold_index]
amplitude_threshold = np.exp(log_threshold)

amplitude_threshold

In [ ]:
# Obeležavanje ROI ispod granice
analysis_df["weak_k_below_gmm_threshold"] = (
    analysis_df["k_ref_amplitude"]
    < amplitude_threshold
)

analysis_df["weak_k_below_gmm_threshold"].value_counts()

In [ ]:
# Vizuelizacija
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(
    analysis_df["k_ref_amplitude"],
    bins=40,
    edgecolor="black",
)

ax.axvline(
    amplitude_threshold,
    linestyle="--",
    color="red",
    label=f"GMM granica = {amplitude_threshold:.3f}",
)

ax.set_xlabel("K+ amplituda")
ax.set_ylabel("Broj ROI")
ax.set_title("Granična vrednost slabog K+ odgovora")
ax.legend()
fig.tight_layout()

save_figure(
    fig,
    "gmm_absolute_threshold",
)

plt.show()

In [ ]:
# Udeo slabih odgovora po eksperimentu
weak_k_by_experiment = (
    analysis_df
    .groupby(
        [
            "experiment_source_id",
            "patient_id",
            "class_label",
        ]
    )
    .agg(
        n_rois=("record_id", "count"),
        n_weak_gmm=(
            "weak_k_below_gmm_threshold",
            "sum",
        ),
        fraction_weak_gmm=(
            "weak_k_below_gmm_threshold",
            "mean",
        ),
        n_weak_component_90=(
            "weak_k_component_flag_90",
            "sum",
        ),
        fraction_weak_component_90=(
            "weak_k_component_flag_90",
            "mean",
        ),
    )
    .reset_index()
)

weak_k_by_experiment.sort_values("fraction_weak_gmm")

Rezultati dvokomponentnog modela nedvosmisleno potvrđuju zaključak da je multimodalnost prisutna u ovom setu podataka uglavnom poreklom od razlika između pojedinačnih eksperimenata. Sa procenjenom graničnom vrednošću amplitude (`2.441`), udeo ćelija koje bi se u datom eksperimentu označile kao slabe kreće se od `0.01` do `0.99`, a bilo bi isključeno oko `28%` svih ćelija, pri čemu bi se za čak 8 eksperimenata odbacilo više od polovine svih ROI. Čak i sa primenom konzervativnog pravila isključenja (npr. najslabijih `2.5%` ćelija), ne bismo govorili o bilo kakvom biološki relevantnom kriterijumu, već o arbitrarnoj granici koja bi i dalje imala nesrazmerne efekte među različitim eksperimentima.

Model Gausove mešavine koji je implementiran suštinski kombinuje dva potencijalna razloga zašto bi neke ćelije pokazivale slab odgovor. Prvi se odnosi na problem na nivou pojedinačnog eksperimenta, koji bi mogao biti uzrokovan razlikama u pripremi čitavog preparata, u uslovima snimanja, kvaliteta ćelija i primenjenih rastvora i slično. Drugi razlog se odnosi na problem na nivou pojedinačne ćelije, koji bi mogao biti izazvan poremećajima u ćelijskoj fiziologiji, neadekvatnim obeležavanjem fluorescentnom bojom, lošom identifikacijom ROI na snimku, itd. Na osnovu toga se može reći da je pre analize rezultata svakog snimka potrebno vršiti kontrolu kvaliteta rezultata na oba nivoa. 

Ono što možemo isprobati u svrhu identifikacije ROI koji nisu validni jeste ispitivanje relativne amplitude odgovora na K+ za svaki ROI na nivou eksperimenta. Ovo u principu ne pomaže u rešavanju originalnog problema (identifikacije relativno pouzdane granične vrednosti K+ amplitude za buduće eksperimente), ali može pomoći u identifikaciji mogućih autlajera za svaki pojedinačni eksperiment.

### Normalizacija K+ odgovora po eksperimentu

In [ ]:
# Definisanje relativnih amplituda
analysis_df["experiment_median_k_amplitude"] = (
    analysis_df
    .groupby("experiment_source_id")["k_ref_amplitude"]
    .transform("median")
)

analysis_df["relative_k_amplitude"] = (
    analysis_df["k_ref_amplitude"]
    / analysis_df["experiment_median_k_amplitude"]
)

analysis_df["log_relative_k_amplitude"] = np.log(
    analysis_df["relative_k_amplitude"]
)

In [ ]:
# Provera raspodele
analysis_df["relative_k_amplitude"].describe(
    percentiles=[
        0.01,
        0.025,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.975,
        0.99,
    ]
)

In [ ]:
# Vizuelizacija raspodele
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(
    analysis_df["relative_k_amplitude"],
    bins=50,
    edgecolor="black",
)

ax.axvline(
    1.0,
    linestyle="--",
    color="red",
    label="Eksperimentalna medijana",
)

ax.set_xlabel("Normalizovana K+ amplituda")
ax.set_ylabel("Broj ROI")
ax.set_title("K+ amplituda normalizovana u okviru svakog eksperimenta")
ax.legend()
fig.tight_layout()

save_figure(
    fig,
    "relative_k_amplitude_distribution",
)

plt.show()

In [ ]:
# Log-transformacija
plt.figure(figsize=(7, 4))

plt.hist(
    analysis_df["log_relative_k_amplitude"],
    bins=50,
    edgecolor="black",
)

plt.axvline(
    0.0,
    linestyle="--",
    color="red",
    label="Eksperimentalna medijana",
)

plt.xlabel("log(K+ amplituda)")
plt.ylabel("Broj ROI")
plt.title("Log-normalizovana raspodela K+ odgovora")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# GMM sa normalizovanim vrednostima
X_relative = (
    analysis_df["log_relative_k_amplitude"]
    .dropna()
    .to_numpy()
    .reshape(-1, 1)
)

relative_gmm_rows = []

for n_components in [1, 2, 3]:
    gmm = GaussianMixture(
        n_components=n_components,
        random_state=RANDOM_STATE,
    )

    gmm.fit(X_relative)

    relative_gmm_rows.append(
        {
            "n_components": n_components,
            "bic": gmm.bic(X_relative),
            "aic": gmm.aic(X_relative),
        }
    )

relative_gmm_comparison = pd.DataFrame(
    relative_gmm_rows
)

relative_gmm_comparison

In [ ]:
# Fitovanje dvokomponentnog modela
relative_gmm_2 = GaussianMixture(
    n_components=2,
    random_state=RANDOM_STATE,
)

relative_gmm_2.fit(X_relative)

In [ ]:
# Kriva gustine
x_grid = np.linspace(
    X_relative.min(),
    X_relative.max(),
    1000,
).reshape(-1, 1)

component_means = relative_gmm_2.means_.ravel()
component_vars = relative_gmm_2.covariances_.ravel()
component_sds = np.sqrt(component_vars)
component_weights = relative_gmm_2.weights_.ravel()

component_densities = []

for mean, sd, weight in zip(
    component_means,
    component_sds,
    component_weights,
):
    density = (
        weight
        * (1 / (sd * np.sqrt(2 * np.pi)))
        * np.exp(
            -0.5
            * ((x_grid.ravel() - mean) / sd) ** 2
        )
    )

    component_densities.append(density)

total_density = np.sum(
    component_densities,
    axis=0,
)

In [ ]:
# Vizuelizacija
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(
    X_relative.ravel(),
    bins=40,
    density=True,
    alpha=0.5,
    edgecolor="black",
    label="Logaritam zabeleženih amplituda",
)

ax.plot(
    x_grid.ravel(),
    total_density,
    linewidth=2,
    label="Dvokomponentni GMM",
)

for i, density in enumerate(component_densities):
    ax.plot(
        x_grid.ravel(),
        density,
        linestyle="--",
        label=f"Komponenta {i + 1}",
    )

ax.axvline(
    0, 
    linestyle="--",
    color="purple",
    label="Eksperimentalna medijana",
)
ax.set_xlabel("log(relativna K+ amplituda)")
ax.set_ylabel("Gustina")
ax.set_title("Model raspodele relativne K+ amplitude kao Gausove mešavine")
ax.legend()
fig.tight_layout()

save_figure(
    fig,
    "gmm_relative_k_amplitude_distribution",
)

plt.show()

In [ ]:
# Identifikacija komponente sa slabim odgovorom
relative_low_component = np.argmin(
    relative_gmm_2.means_.ravel()
)

relative_component_probabilities = relative_gmm_2.predict_proba(X_relative)

relative_low_component_probability = (
    relative_component_probabilities[:, relative_low_component]
)

# Dodavanje vrednosti u tabelu
analysis_df["relative_weak_k_component_probability"] = (
    relative_low_component_probability
)

analysis_df["relative_weak_k_component_flag_90"] = (
    analysis_df["relative_weak_k_component_probability"] >= 0.90
)

analysis_df["relative_weak_k_component_flag_75"] = (
    analysis_df["relative_weak_k_component_probability"] >= 0.75
)

In [ ]:
# Provera raspodele verovatnoće pripadanja slaboj komponenti
analysis_df[
    [
        "relative_weak_k_component_probability",
    ]
].describe()

In [ ]:
# Broj ROI s datom verovatnoćom pripadanja slaboj komponenti
analysis_df[
    [
        "relative_weak_k_component_flag_90",
        "relative_weak_k_component_flag_75",
    ]
].sum()

In [ ]:
# Procena granične vrednosti
relative_grid_probabilities = relative_gmm_2.predict_proba(x_grid)

relative_low_component_grid_probability = (
    relative_grid_probabilities[:, relative_low_component]
)

# Tačka sa 50% verovatnoćom pripadanja obema komponentama
relative_threshold_index = np.argmin(
    np.abs(relative_low_component_grid_probability - 0.5)
)

relative_log_threshold = x_grid.ravel()[relative_threshold_index]
relative_amplitude_threshold = np.exp(relative_log_threshold)

relative_amplitude_threshold

In [ ]:
# Obeležavanje ROI ispod granice
analysis_df["relative_weak_k_below_gmm_threshold"] = (
    analysis_df["relative_k_amplitude"]
    < relative_amplitude_threshold
)

analysis_df["relative_weak_k_below_gmm_threshold"].value_counts()

In [ ]:
# Vizuelizacija
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(
    analysis_df["relative_k_amplitude"],
    bins=40,
    edgecolor="black",
)

ax.axvline(
    relative_amplitude_threshold,
    linestyle="--",
    color="red",
    label=f"GMM granica = {relative_amplitude_threshold:.3f}",
)

ax.set_xlabel("Normalizovana K+ amplituda")
ax.set_ylabel("Broj ROI")
ax.set_title("Granična vrednost slabog normalizovanog K+ odgovora za svaki eksperiment")
ax.legend()
fig.tight_layout()

save_figure(
    fig,
    "gmm_relative_threshold",
)

plt.show()

In [ ]:
# Svi GMM rezultati
gmm_threshold_summary = pd.DataFrame(
    [
        {
            "analysis": (
                "absolute_k_amplitude"
            ),
            "threshold_type": (
                "two_component_gmm_"
                "equal_probability"
            ),
            "threshold": (
                amplitude_threshold
            ),
            "n_below_threshold": int(
                analysis_df[
                    "weak_k_below_gmm_threshold"
                ].sum()
            ),
            "fraction_below_threshold": float(
                analysis_df[
                    "weak_k_below_gmm_threshold"
                ].mean()
            ),
        },
        {
            "analysis": (
                "relative_k_amplitude"
            ),
            "threshold_type": (
                "two_component_gmm_"
                "equal_probability"
            ),
            "threshold": (
                relative_amplitude_threshold
            ),
            "n_below_threshold": int(
                analysis_df[
                    (
                        "relative_weak_k_"
                        "below_gmm_threshold"
                    )
                ].sum()
            ),
            "fraction_below_threshold": float(
                analysis_df[
                    (
                        "relative_weak_k_"
                        "below_gmm_threshold"
                    )
                ].mean()
            ),
        },
    ]
)

gmm_threshold_summary

Na osnovu rezultata GMM, vidimo da i nakon normalizacije amplituda, raspodela vrednosti zadržava izvesnu asimetriju ili više pojedinačnih regiona. Naravno, to i dalje ne znači da postoje dva, tri ili više tipa ćelija u svakom eksperimentu - jednostavno nam govori da se postojeća raspodela može matematički bolje okarakterisati većim brojem pojedinačnih komponenti. Svakako ne možemo govoriti o dve relativno dobro definisane populacije ćelija, jer vidimo da je preklapanje raspodela u GMM dvokomponentnom modelu veoma izraženo - medijalna verovatnoća pripadanja niskoj komponenti na nivou ROI je `0.155`, dok je minimalna verovatnoća čak `0.121`. Dakle, zaključujemo da verovatnoće zauzimaju kontinuum vrednosti i da je teško identifikovati određenu graničnu vrednost, čak i u okviru pojedinačnih eksperimenata.

Ono što dalje možemo ispitati jeste postojanje autlajera u okviru svakog eksperimenta. Autlajeri se mogu identifikovati preko **z-vrednosti**, koja govori koliko se neka vrednost razlikuje od srednje vrednosti u jedinicama standardne devijacije:

\begin{equation}
    z = \frac{x - \overline{x}}{s}
\end{equation}

Međutim, ova mera je osetljiva na prisustvo ekstremnih vrednosti, pa je bolje koristiti **robusnu z-vrednost**, koja umesto srednje vrednosti koristi medijanu i medijalnu apsolutnu devijaciju (MAD), koja se računa kao:

\begin{equation}
    MAD = median(|x_i - median(x)|)
\end{equation}

i koja daje meru apsolutne razlike svakog zapažanja i medijalne vrednosti. Robusna z-vrednost se računa kao:

\begin{equation}
    z_{robust} = 0.6745 \frac{x - median(x)}{MAD}
\end{equation}

In [ ]:
# Procena autlajera u okviru svakog eksperimenta
def add_robust_k_scores(group):
    values = group["k_ref_amplitude"]

    median = values.median()

    mad = np.median(
        np.abs(values - median)
    )

    group = group.copy()

    if mad == 0 or np.isnan(mad):
        group["k_robust_z"] = np.nan
    else:
        group["k_robust_z"] = (
            0.6745
            * (values - median)
            / mad
        )

    return group

In [ ]:
# Dodavanje vrednosti u tabelu
analysis_df = (
    analysis_df
    .groupby(
        "experiment_source_id",
        group_keys=False,
    )
    .apply(add_robust_k_scores)
)

In [ ]:
# Definisanje dve granične vrednosti
analysis_df["weak_k_robust_flag_3"] = (
    analysis_df["k_robust_z"] < -3
)

analysis_df["weak_k_robust_flag_2_5"] = (
    analysis_df["k_robust_z"] < -2.5
)

In [ ]:
# Prikaz rezultata
robust_weak_summary = (
    analysis_df
    .groupby("experiment_label")
    .agg(
        n_rois=("record_id", "count"),
        median_k_amplitude=(
            "k_ref_amplitude",
            "median",
        ),
        n_weak_z3=(
            "weak_k_robust_flag_3",
            "sum",
        ),
        fraction_weak_z3=(
            "weak_k_robust_flag_3",
            "mean",
        ),
        n_weak_z2_5=(
            "weak_k_robust_flag_2_5",
            "sum",
        ),
        fraction_weak_z2_5=(
            "weak_k_robust_flag_2_5",
            "mean",
        ),
    )
    .reset_index()
)

robust_weak_summary

In [ ]:
# Vizuelizacija raspodele robusnih z-vrednosti
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(
    analysis_df["k_robust_z"].dropna(),
    bins=60,
    edgecolor="black",
)

ax.axvline(
    -2.5,
    linestyle="--",
    color="purple",
    label="Potencijalna granica: z = −2.5",
)

ax.axvline(
    -3.0,
    linestyle=":",
    color="purple",
    label="Potencijalna granica: z = −3",
)

ax.axvline(
    0,
    linestyle="-",
    color="red",
    label="Eksperimentalna medijana",
)

ax.set_xlabel("Robusna z-vrednost u eksperimentu")
ax.set_ylabel("Broj ROI")
ax.set_title("Ukupna raspodela z-vrednosti K+ amplitude")
ax.legend()
fig.tight_layout()

save_figure(
    fig,
    "robust_z_distribution",
)

plt.show()

In [ ]:
# Vizuelizacija udela obeleženih ROI po eksperimentu
robust_plot_data = (
    robust_weak_summary
    .sort_values(
        "fraction_weak_z2_5",
        ascending=True,
    )
    .reset_index(drop=True)
)

y_positions = np.arange(
    len(robust_plot_data)
)

bar_height = 0.38

plt.figure(figsize=(12,10))

plt.barh(
    y_positions - bar_height / 2,
    robust_plot_data["fraction_weak_z2_5"],
    height=bar_height,
    label="Robusno z < −2.5",
)

plt.barh(
    y_positions + bar_height / 2,
    robust_plot_data["fraction_weak_z3"],
    height=bar_height,
    label="Robusno z < −3",
)

plt.yticks(
    y_positions,
    robust_plot_data["experiment_label"],
)

plt.xlabel("Udeo obeleženih ROI")
plt.ylabel("Eksperiment")
plt.title("Potencijalni slabi odgovori na K+ po eksperimentu")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Zajednički prikaz normalizovane raspodele i autlajera po eksperimentu
experiment_order = (
    robust_weak_summary
    .sort_values("fraction_weak_z3")
    ["experiment_label"]
    .tolist()
)

boxplot_data = [
    analysis_df.loc[
        analysis_df["experiment_label"]
        == experiment,
        "relative_k_amplitude",
    ].dropna()
    for experiment in experiment_order
]

fig, ax = plt.subplots(figsize=(12, 10))

ax.boxplot(
    boxplot_data,
    tick_labels=experiment_order,
    orientation="horizontal",
    showfliers=False,
)

label_added = False

for position, experiment in enumerate(
    experiment_order,
    start=1,
):
    flagged_values = analysis_df.loc[
        (
            analysis_df["experiment_label"]
            == experiment
        )
        & analysis_df["weak_k_robust_flag_3"],
        "relative_k_amplitude",
    ]

    if not flagged_values.empty:
        ax.scatter(
            flagged_values,
            np.repeat(position, len(flagged_values)),
            marker="x",
            label=(
                "Robusno z < −3"
                if not label_added
                else None
            ),
        )

        label_added = True

ax.axvline(
    1,
    linestyle="--",
    color="red",
    label="Eksperimentalna medijana",
)

ax.set_xlabel("Relativna K+ amplituda")
ax.set_ylabel("Eksperiment")
ax.set_title("Raspodela normalizovane K+ amplitude i potencijalni slabi odgovori")

if label_added:
    ax.legend()

fig.tight_layout()

save_figure(
    fig,
    "relative_k_amplitude_with_outliers",
)

plt.show()

Nakon ispitivanja z-vrednosti, dobili smo sasvim drugačije rezultate nego prilikom modeliranja raspodele apsolutnih vrednosti K+ amplitude pomoću GMM. Korišćenje z-vrednosti za identifikaciju autlajera obeležava svega `19`, odnosno `54` ROI kao slabe u zavisnosti od odabrane granične vrednosti. Pritom, svaki pojedinačni eksperiment ima mali udeo obeleženih ROI. Ovo nam ponovo ilustruje da su ROI sa niskom amplitudom uglavnom poreklom od eksperimenata koji u celosti pokazuju nisku amplitudu odgovora. 

#### Zaključak

Na osnovu urađenih analiza, zaključujemo da amplitude odgovora na K+ među ćelijama imaju heterogenu i multimodalnu raspodelu, kao i da se opseg amplituda među ćelijama značajno razlikuje među eksperimentima. Kada se posmatraju sve ćelije zajedno, uočava se da ćelije sa slabijim odgovorom gotovo univerzalno potiču od eksperimenata koji se u celosti odlikuju nižom postignutom amplitudom odgovora na K+. Unutar pojedinačnih eksperimenata, nisu identifikovane značajne populacije ćelija sa nekarakteristično niskim odgovorom. 

Postavljanje univerzalne granice K+ amplitude moglo bi diskvalifikovati veliki udeo ROI iz određenih eksperimenata, a bez informacija o mogućim razlozima zbog kojih se ćelije u pomenutim eksperimentima ponašaju na taj način, i na osnovu trenutno dostupnih podataka, teško je opravdati uvođenje ovakve granice. 

In [ ]:
# Čuvanje konačne tabele korišćene za analizu
K_FINAL_TABLE_PATH = (
    K_EXTRACTED_DIR
    / "k_first_amplitude_analysis_final.csv"
)

analysis_df.to_csv(
    K_FINAL_TABLE_PATH,
    index=False,
)

In [ ]:
# Čuvanje rezultata
result_tables = {
    "k_file_summary.csv": (
        k_file_summary
    ),
    "k_dilution_summary.csv": (
        k_dilution_summary
    ),
    "k_candidate_thresholds.csv": (
        candidate_thresholds
    ),
    "k_absolute_gmm_comparison.csv": (
        gmm_comparison
    ),
    "k_relative_gmm_comparison.csv": (
        relative_gmm_comparison
    ),
    "k_gmm_threshold_summary.csv": (
        gmm_threshold_summary
    ),
    "weak_k_candidates_by_experiment.csv": (
        weak_k_by_experiment
    ),
    "k_robust_outlier_summary.csv": (
        robust_weak_summary
    ),
}

for file_name, table in (
    result_tables.items()
):
    output_path = (
        K_TABLES_DIR
        / file_name
    )

    table.to_csv(
        output_path,
        index=False,
    )